In [1]:
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
from pathlib import Path

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, Model

from sklearn.preprocessing import LabelEncoder, MinMaxScaler
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.tree import DecisionTreeClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.svm import LinearSVC
from sklearn.calibration import CalibratedClassifierCV
from sklearn.metrics import f1_score, roc_auc_score, precision_score, recall_score, classification_report

SEED = 42
np.random.seed(SEED)
tf.random.set_seed(SEED)

COL_NAMES = [
    'duration', 'protocol_type', 'service', 'flag',
    'src_bytes', 'dst_bytes', 'land', 'wrong_fragment', 'urgent', 'hot',
    'num_failed_logins', 'logged_in', 'num_compromised', 'root_shell',
    'su_attempted', 'num_root', 'num_file_creations', 'num_shells',
    'num_access_files', 'num_outbound_cmds', 'is_host_login',
    'is_guest_login', 'count', 'srv_count', 'serror_rate',
    'srv_serror_rate', 'rerror_rate', 'srv_rerror_rate',
    'same_srv_rate', 'diff_srv_rate', 'srv_diff_host_rate',
    'dst_host_count', 'dst_host_srv_count', 'dst_host_same_srv_rate',
    'dst_host_diff_srv_rate', 'dst_host_same_src_port_rate',
    'dst_host_srv_diff_host_rate', 'dst_host_serror_rate',
    'dst_host_srv_serror_rate', 'dst_host_rerror_rate',
    'dst_host_srv_rerror_rate', 'label', 'difficulty'
]

SELECTED_FEATURES = [
    'src_bytes', 'service', 'dst_bytes', 'flag',
    'diff_srv_rate', 'same_srv_rate', 'dst_host_srv_count', 'dst_host_same_srv_rate',
]

def find_file(filename):
    search_paths = [
        (Path.home() / ".cache" / "kagglehub" / "datasets" / "hassan06" / "nslkdd" / "versions" / "1"),
        Path("."),
        Path.home() / "Downloads",
    ]
    for base in search_paths:
        candidate = base / filename
        if candidate.exists():
            return candidate
    raise FileNotFoundError(f"No se encontro '{filename}'.")

df = pd.read_csv(find_file('KDDTrain+_20Percent.txt'), header=None, names=COL_NAMES).drop(columns=['difficulty'])
print(f"Instancias: {len(df):,}")

Instancias: 25,192


In [2]:
protocol_map = {'tcp': 1, 'udp': 2, 'icmp': 3}
df['protocol_type'] = df['protocol_type'].str.lower().map(protocol_map).fillna(0)
df['service'] = LabelEncoder().fit_transform(df['service'].astype(str))
df['flag']    = LabelEncoder().fit_transform(df['flag'].astype(str))

DOS   = {'back','land','neptune','pod','smurf','teardrop','apache2','udpstorm','processtable','mailbomb'}
PROBE = {'ipsweep','nmap','portsweep','satan','mscan','saint'}
R2L   = {'ftp_write','guess_passwd','imap','multihop','phf','spy','warezclient','warezmaster',
         'sendmail','named','snmpgetattack','snmpguess','xlock','xsnoop','worm'}
U2R   = {'buffer_overflow','loadmodule','perl','rootkit','httptunnel','ps','sqlattack','xterm'}

def map_label(label):
    label = label.lower().strip()
    if label == 'normal': return 1
    if label in DOS:      return 2
    if label in PROBE:    return 3
    if label in R2L:      return 4
    if label in U2R:      return 5
    return 2

df['label'] = df['label'].apply(map_label)

X = df[SELECTED_FEATURES].values.astype(np.float32)
y = df['label'].values

X_tv, X_test, y_tv, y_test = train_test_split(X, y, test_size=0.20, random_state=SEED, stratify=y)
X_tr, X_val, y_tr, y_val   = train_test_split(X_tv, y_tv, test_size=0.20, random_state=SEED, stratify=y_tv)

# CORREGIDO: scaler ajustado solo sobre train final (post-split), sin data leakage
scaler  = MinMaxScaler()
X_tr    = scaler.fit_transform(X_tr)
X_val   = scaler.transform(X_val)
X_test  = scaler.transform(X_test)

CLASS_NAMES = {1: 'Normal', 2: 'DoS', 3: 'Probe', 4: 'R2L', 5: 'U2R'}
N_FEATURES  = X_tr.shape[1]

print(f"Train: {X_tr.shape} | Val: {X_val.shape} | Test: {X_test.shape}")

Train: (16122, 8) | Val: (4031, 8) | Test: (5039, 8)


In [3]:
def build_autoencoder(input_dim, latent_dim=3):
    inputs = keras.Input(shape=(input_dim,))
    x = layers.Dense(32, activation='tanh')(inputs)
    x = layers.Dense(24, activation='tanh')(x)
    x = layers.Dense(12, activation='tanh')(x)
    x = layers.Dense(6,  activation='tanh')(x)
    latent  = layers.Dense(latent_dim, activation='tanh', name='latent')(x)
    x = layers.Dense(6,  activation='tanh')(latent)
    x = layers.Dense(12, activation='tanh')(x)
    x = layers.Dense(24, activation='tanh')(x)
    x = layers.Dense(32, activation='tanh')(x)
    outputs = layers.Dense(input_dim, activation='linear')(x)
    autoencoder = Model(inputs, outputs, name='AE_Van2020')
    encoder     = Model(inputs, latent,  name='Encoder_Van2020')
    return autoencoder, encoder

autoencoder, encoder = build_autoencoder(input_dim=N_FEATURES, latent_dim=3)
autoencoder.compile(optimizer=keras.optimizers.Adam(learning_rate=0.001), loss='mse')

early_stop = keras.callbacks.EarlyStopping(monitor='val_loss', patience=10, restore_best_weights=True, verbose=0)
autoencoder.fit(
    X_tr, X_tr,
    validation_data=(X_val, X_val),
    epochs=50, batch_size=256,
    callbacks=[early_stop], verbose=1
)

Z_train = encoder.predict(X_tr,   verbose=0)
Z_val   = encoder.predict(X_val,  verbose=0)
Z_test  = encoder.predict(X_test, verbose=0)
print(f"Espacio latente — Train: {Z_train.shape} | Val: {Z_val.shape} | Test: {Z_test.shape}")

scoring = 'f1_weighted'

dt_gs = GridSearchCV(
    DecisionTreeClassifier(random_state=SEED),
    param_grid={'max_depth': [3, 5, 10, 15, None], 'min_samples_split': [2, 5, 10], 'criterion': ['gini', 'entropy']},
    cv=5, scoring=scoring, n_jobs=-1
)
dt_gs.fit(Z_train, y_tr)
best_dt = dt_gs.best_estimator_
print(f"DT  — mejores params: {dt_gs.best_params_} | CV F1: {dt_gs.best_score_:.4f}")

knn_gs = GridSearchCV(
    KNeighborsClassifier(),
    param_grid={'n_neighbors': [3, 5, 7, 10, 15], 'metric': ['euclidean', 'manhattan'], 'weights': ['uniform', 'distance']},
    cv=5, scoring=scoring, n_jobs=-1
)
knn_gs.fit(Z_train, y_tr)
best_knn = knn_gs.best_estimator_
print(f"kNN — mejores params: {knn_gs.best_params_} | CV F1: {knn_gs.best_score_:.4f}")

best_f1, best_svm, best_C = -1, None, None
for C in [0.01, 0.1, 1.0, 10.0]:
    clf = CalibratedClassifierCV(LinearSVC(C=C, max_iter=2000, random_state=SEED), cv=3)
    clf.fit(Z_train, y_tr)
    f1 = f1_score(y_val, clf.predict(Z_val), average='weighted')
    if f1 > best_f1:
        best_f1, best_svm, best_C = f1, clf, C
print(f"SVM — mejor C: {best_C} | F1 val: {best_f1*100:.2f}%")

classifiers = {'DT': best_dt, 'kNN': best_knn, 'SVM': best_svm}

Epoch 1/50



 1/63 ━━━━━━━━━━━━━━━━━━━━ 2:27 2s/step - loss: 0.2002


20/63 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - loss: 0.1202 


38/63 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - loss: 0.0966


56/63 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - loss: 0.0833

...


63/63 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - loss: 0.0023 - val_loss: 0.0023


Espacio latente — Train: (16122, 3) | Val: (4031, 3) | Test: (5039, 3)


DT  — mejores params: {'criterion': 'entropy', 'max_depth': 15, 'min_samples_split': 2} | CV F1: 0.9508


kNN — mejores params: {'metric': 'euclidean', 'n_neighbors': 3, 'weights': 'distance'} | CV F1: 0.9620


SVM — mejor C: 0.01 | F1 val: 80.19%


In [4]:
CLASSES      = sorted(np.unique(y_test))
target_names = [CLASS_NAMES[c] for c in CLASSES]

SEP = '=' * 58
print("Autoencoder + DT/kNN/SVM — Dataset NSL-KDD")
print(SEP)
print(f"{'Modelo':<6} {'Prec':>6} {'Rec':>6} {'F1':>6}")
print(f"{'─'*6} {'─'*6} {'─'*6} {'─'*6}")

for name, clf in classifiers.items():
    y_pred = clf.predict(Z_test)
    prec = precision_score(y_test, y_pred, average='weighted', zero_division=0)
    rec  = recall_score   (y_test, y_pred, average='weighted', zero_division=0)
    f1   = f1_score       (y_test, y_pred, average='weighted', zero_division=0)
    print(f"{name:<6} {prec:.3f}  {rec:.3f}  {f1:.3f}")

print(SEP)
print(classification_report(y_test, best_dt.predict(Z_test), labels=CLASSES, target_names=target_names, zero_division=0))

Autoencoder + DT/kNN/SVM — Dataset NSL-KDD
Modelo   Prec    Rec     F1
────── ────── ────── ──────
DT     0.951  0.951  0.950
kNN    0.962  0.962  0.962
SVM    0.781  0.845  0.801
              precision    recall  f1-score   support

      Normal       0.96      0.96      0.96      2690
         DoS       0.96      0.96      0.96      1847
       Probe       0.86      0.91      0.89       458
         R2L       0.64      0.50      0.56        42
         U2R       0.50      0.50      0.50         2

    accuracy                           0.95      5039
   macro avg       0.78      0.77      0.77      5039
weighted avg       0.95      0.95      0.95      5039

